# Hybrid Graph-Based Recommender (Neo4j)

Idea: store every user and every anime as a node in Neo4j with its embedding as a property. Edges are `(:User)-[:RATED {rating}]->(:Anime)`.

Recommendation = **2-hop graph traversal**:
1. From the target user, use Neo4j's vector index to find top-N similar users (cosine over user vectors).
2. From that pool, follow `RATED` edges, exclude animes the target already saw, aggregate `user_similarity * normalized_rating`, rank.

## 1. Setup

In [38]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import faiss
from neo4j import GraphDatabase

load_dotenv()

True

In [39]:
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USER     = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("connected")

connected


In [21]:
DATA_DIR     = Path.cwd().parent / "data"
ARTIFACT_DIR = Path.cwd().parent / "faiss_artifacts"

SAMPLE_USERS    = 1000   # Aura Free caps at 400k relationships total
MIN_RATINGS     = 20     # only sample users with at least this many ratings
SIMILAR_USERS_K = 50     # 1st hop: how many similar users to walk to
TOP_K           = 10     # final number of recommendations
RANDOM_SEED     = 42

In [22]:
# One-time cleanup: wipe partial RATED edges + nodes from the previous oversized run.
# Run this once before re-ingesting with the smaller sample.
with driver.session() as s:
    s.run("MATCH ()-[r:RATED]->() CALL { WITH r DELETE r } IN TRANSACTIONS OF 10000 ROWS").consume()
    s.run("MATCH (u:User)  CALL { WITH u DETACH DELETE u } IN TRANSACTIONS OF 5000 ROWS").consume()
    s.run("MATCH (a:Anime) CALL { WITH a DETACH DELETE a } IN TRANSACTIONS OF 5000 ROWS").consume()
print("wiped")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (r) { ... }', position=<SummaryInputPosition line=1, column=24, offset=23>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 23, 'line': 1, 'column': 24}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:RATED]->() CALL { WITH r DELETE r } IN TRANSACTIONS OF 10000 ROWS'
Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (u) { ... }', position=<SummaryInputPosition line=1, column=17, offset

wiped


## 2. Load artifacts (anime vectors, user vectors, ratings)

In [23]:
# Anime vectors: reconstruct from the content FAISS index
with open(ARTIFACT_DIR / "mal_id_to_faiss_id.json", "r", encoding="utf-8") as f:
    mal_id_to_faiss_id = {int(k): int(v) for k, v in json.load(f).items()}

content_index = faiss.read_index(str(ARTIFACT_DIR / "content_faiss_index.bin"))
EMBED_DIM = content_index.d

anime_id_to_vec = {
    mal_id: content_index.reconstruct(faiss_id).astype("float32")
    for mal_id, faiss_id in mal_id_to_faiss_id.items()
}
print(f"anime vectors: {len(anime_id_to_vec)}, dim={EMBED_DIM}")

anime vectors: 16206, dim=256


In [24]:
# User vectors
with open(ARTIFACT_DIR / "user_id_to_vector.json", "r", encoding="utf-8") as f:
    user_id_to_vector = {int(k): np.asarray(v, dtype="float32") for k, v in json.load(f).items()}

print(f"user vectors: {len(user_id_to_vector)}")

user vectors: 309484


In [25]:
# Ratings
ratings_df = pd.read_csv(DATA_DIR / "rating_complete.csv")
ratings_df = ratings_df[ratings_df["rating"] > 0]
ratings_df = ratings_df[ratings_df["anime_id"].isin(anime_id_to_vec)]
ratings_df = ratings_df[ratings_df["user_id"].isin(user_id_to_vector)]
len(ratings_df)

56726814

In [26]:
# Sample users (must have at least MIN_RATINGS ratings so 2-hop has signal)
counts = ratings_df.groupby("user_id").size()
eligible = counts[counts >= MIN_RATINGS].index.to_numpy()
rng = np.random.default_rng(RANDOM_SEED)
sample_user_ids = rng.choice(eligible, size=min(SAMPLE_USERS, len(eligible)), replace=False)

sample_ratings = ratings_df[ratings_df["user_id"].isin(sample_user_ids)].copy()
sample_anime_ids = sample_ratings["anime_id"].unique()
print(f"sampled {len(sample_user_ids)} users, {len(sample_anime_ids)} animes, {len(sample_ratings)} ratings")

sampled 1000 users, 8439 animes, 206423 ratings


In [27]:
# Titles for nice output
anime_titles = (
    pd.read_csv(DATA_DIR / "anime.csv", usecols=["MAL_ID", "Name"])
    .set_index("MAL_ID")["Name"]
    .to_dict()
)

## 3. Schema: constraints + vector indexes

Vector indexes need Neo4j 5.11+ (Aura is fine). Cosine matches how user/anime vectors were normalized.

In [28]:
def run(cypher, **params):
    with driver.session() as s:
        return s.run(cypher, **params).data()

run("CREATE CONSTRAINT user_id  IF NOT EXISTS FOR (u:User)  REQUIRE u.id     IS UNIQUE")
run("CREATE CONSTRAINT anime_id IF NOT EXISTS FOR (a:Anime) REQUIRE a.mal_id IS UNIQUE")

run(f"""
CREATE VECTOR INDEX user_vec  IF NOT EXISTS
FOR (u:User)  ON (u.vector)
OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}
""")
run(f"""
CREATE VECTOR INDEX anime_vec IF NOT EXISTS
FOR (a:Anime) ON (a.vector)
OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}
""")
print("schema ready")

schema ready


## 4. Ingest nodes and edges

Batched UNWIND for speed. Re-running is safe (MERGE + idempotent indexes).

In [29]:
def batched(iterable, size):
    buf = []
    for x in iterable:
        buf.append(x)
        if len(buf) >= size:
            yield buf
            buf = []
    if buf:
        yield buf

In [30]:
# Anime nodes
anime_rows = [
    {"mal_id": int(mid), "vector": anime_id_to_vec[int(mid)].tolist(), "title": anime_titles.get(int(mid), str(mid))}
    for mid in sample_anime_ids
]

for chunk in batched(anime_rows, 500):
    run(
        """
        UNWIND $rows AS row
        MERGE (a:Anime {mal_id: row.mal_id})
        SET a.title = row.title
        WITH a, row
        CALL db.create.setNodeVectorProperty(a, 'vector', row.vector)
        """,
        rows=chunk,
    )
print(f"loaded {len(anime_rows)} anime nodes")

loaded 8439 anime nodes


In [31]:
# User nodes
user_rows = [
    {"id": int(uid), "vector": user_id_to_vector[int(uid)].tolist()}
    for uid in sample_user_ids
]

for chunk in batched(user_rows, 500):
    run(
        """
        UNWIND $rows AS row
        MERGE (u:User {id: row.id})
        WITH u, row
        CALL db.create.setNodeVectorProperty(u, 'vector', row.vector)
        """,
        rows=chunk,
    )
print(f"loaded {len(user_rows)} user nodes")

loaded 1000 user nodes


In [32]:
# RATED edges
edge_rows = sample_ratings[["user_id", "anime_id", "rating"]].to_dict("records")

for i, chunk in enumerate(batched(edge_rows, 5000)):
    run(
        """
        UNWIND $rows AS row
        MATCH (u:User  {id:     row.user_id})
        MATCH (a:Anime {mal_id: row.anime_id})
        MERGE (u)-[r:RATED]->(a)
        SET r.rating = row.rating
        """,
        rows=chunk,
    )
    if (i + 1) % 20 == 0:
        print(f"  {(i+1)*5000} edges...")
print(f"loaded {len(edge_rows)} RATED edges")

  100000 edges...
  200000 edges...
loaded 206423 RATED edges


## 5. Recommend: 2-hop traversal

Cypher does it all in one query:
1. `db.index.vector.queryNodes` on `user_vec` → top-K similar users (with cosine score).
2. Walk their `RATED` edges → collect candidate animes.
3. Exclude animes the target user already rated.
4. Score = `sum(user_similarity * rating/10)`. Rank, return top-K.

In [33]:
RECOMMEND_CYPHER = """
MATCH (target:User {id: $user_id})
CALL db.index.vector.queryNodes('user_vec', $k_users + 1, target.vector)
YIELD node AS peer, score AS user_sim
WHERE peer <> target
WITH target, peer, user_sim
MATCH (peer)-[r:RATED]->(a:Anime)
WHERE NOT EXISTS { (target)-[:RATED]->(a) }
WITH a, sum(user_sim * (r.rating / 10.0)) AS score, count(*) AS support
RETURN a.mal_id AS mal_id, a.title AS title, score, support
ORDER BY score DESC
LIMIT $top_k
"""

def recommend(user_id: int, top_k: int = TOP_K, k_users: int = SIMILAR_USERS_K) -> pd.DataFrame:
    with driver.session() as s:
        rows = s.run(
            RECOMMEND_CYPHER,
            user_id=int(user_id),
            k_users=int(k_users),
            top_k=int(top_k),
        ).data()
    return pd.DataFrame(rows)

In [34]:
# Sanity check
demo_user = int(sample_user_ids[0])
print(f"recommendations for user {demo_user}:")
recommend(demo_user, top_k=10)

recommendations for user 263677:


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=1, offset=36>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 36, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (target:User {id: $user_id})\nCALL db.index.vector.queryNodes('user_vec', $k_users + 1, target.vector)\nYIELD node AS peer, score AS user_sim\nWHERE peer <> target\nWITH target, peer, user_sim\nMATCH (peer)-[r:RATED]->(a:Anime)\nWHERE NOT EXISTS { (target)-[:RATED]->(a) }\nWITH a, sum(user_sim * (r.rating / 10.0)) AS score, count(*

,mal_id,title,score,support
0,9253,Steins;Gate,37.171109,43
1,1575,Code Geass: Hangyaku no Lelouch,35.847939,44
2,2904,Code Geass: Hangyaku no Lelouch R2,32.808863,42
3,4224,Toradora!,32.701905,41
4,9756,Mahou Shoujo Madoka★Magica,32.610451,39
5,6547,Angel Beats!,32.214877,44
6,5114,Fullmetal Alchemist: Brotherhood,32.015673,37
7,5081,Bakemonogatari,31.548840,41
8,199,Sen to Chihiro no Kamikakushi,31.276368,39
9,7311,Suzumiya Haruhi no Shoushitsu,29.070551,34


In [35]:
# Peek at what this user already liked (sanity-check the recs make sense)
with driver.session() as s:
    seen = s.run(
        """
        MATCH (u:User {id: $uid})-[r:RATED]->(a:Anime)
        RETURN a.title AS title, r.rating AS rating
        ORDER BY r.rating DESC LIMIT 10
        """,
        uid=demo_user,
    ).data()
pd.DataFrame(seen)

,title,rating
0,Clannad: After Story,10
1,Kara no Kyoukai 5: Mujun Rasen,10
2,"Elfen Lied: Tooriame nite Arui wa, Shoujo wa I...",10
3,NHK ni Youkoso!,10
4,Elfen Lied,10
5,Monster,10
6,Serial Experiments Lain,10
7,Sakigake!! Cromartie Koukou,10
8,Rozen Maiden: Träumend,10
9,Detroit Metal City,10


## 6. Cleanup

In [36]:
driver.close()

In [40]:
with driver.session() as session:
    user_count = session.run("MATCH (u:User) RETURN count(u)").single()[0]
    anime_count = session.run("MATCH (a:Anime) RETURN count(a)").single()[0]
print(f"Users: {user_count}, Anime: {anime_count}")

Users: 1000, Anime: 8439
